In [ ]:
import pandas as pd
import numpy as np
import re

# Таблица клиентов
customers = pd.DataFrame({
    'Client_ID': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006'],
    'Full_Name': ['Mr. CHARLES DICKENS', '  jane   austen  ', None, 'MRS. MARY SHELLEY', '   leo   tolstoy   ', 'mark twain'],
    'Phone': ['+7 999 123-45-67', 'нет', '8-916-555-33-22', '+44 20 7946 0958', None, '1-800-123-4567'],
    'City': ['Moscow', 'SPb', '  MOSCOW ', 'London', 'Spb', None]
})

# Таблица заказов
orders = pd.DataFrame({
    'Order_ID': ['A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007'],
    'Client_ID': ['C001', 'C002', 'C001', 'C003', 'C005', 'C002', 'C007'],
    'Amount': ['$1,500', '€2.300', '50000 руб.', None, '800', '€1,200', '$999'],
    'Status': ['Done', 'pending', 'DONE', 'CANCELED', 'done', None, 'pending']
})

print("=== Клиенты ===")
print(customers)
print("\n=== Заказы ===")
print(orders)


=== Клиенты ===
  Client_ID            Full_Name             Phone       City
0      C001  Mr. CHARLES DICKENS  +7 999 123-45-67     Moscow
1      C002      jane   austen                 нет        SPb
2      C003                  NaN   8-916-555-33-22    MOSCOW 
3      C004    MRS. MARY SHELLEY  +44 20 7946 0958     London
4      C005     leo   tolstoy                  NaN        Spb
5      C006           mark twain    1-800-123-4567        NaN

=== Заказы ===
  Order_ID Client_ID      Amount    Status
0     A001      C001      $1,500      Done
1     A002      C002      €2.300   pending
2     A003      C001  50000 руб.      DONE
3     A004      C003         NaN  CANCELED
4     A005      C005         800      done
5     A006      C002      €1,200       NaN
6     A007      C007        $999   pending


In [ ]:

def clean_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    x = re.sub(r'\b(mr|mrs)\b\.?\s*', '', x, flags=re.IGNORECASE)
    x = re.sub(r'\s+', ' ', x)
    x = x.strip().title()
    if x == '':
        return np.nan
    return x

df['clean_name'] = customers['Full_Name'].apply(clean_name)
df['clean_name']

0    Charles Dickens
1        Jane Austen
2                NaN
3       Mary Shelley
4        Leo Tolstoy
Name: clean_name, dtype: str

In [ ]:
def clean_phone(x):
    if pd.isna(x):
        return np.nan 
    x = str(x)
    x = re.sub(r'[^0-9]', '', x)
    if x == '':
        return np.nan 
    return x 

df['clean_phone'] = customers['Phone'].apply(clean_phone)
df['clean_phone']

0     79991234567
1             NaN
2     89165553322
3    442079460958
4             NaN
Name: clean_phone, dtype: str

In [ ]:
# %%
# %%
def clean_city(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    x = x.strip()
    if x == 'spb':
        x = x
    if x == '':
        return np.nan
    x = x.title()

    return x 
    
df['clean_city'] = customers['City'].apply(clean_city)
df['clean_city']

0    Moscow
1       Spb
2    Moscow
3    London
4       Spb
Name: clean_city, dtype: str

In [ ]:
# %%
def clean_amounts(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r'[^0-9]', '', x)
    if x == '':
        return np.nan
    return float(x) 

df['clean_amounts'] = orders['Amount'].apply(clean_amounts)
df['clean_amounts']

0     1500.0
1     2300.0
2    50000.0
3        NaN
4      800.0
Name: clean_amounts, dtype: float64

In [ ]:
def clean_status(x):
    if pd.isna(x):
        return np.nan 
    x = str(x).lower()
    if x in ('done'):
        return 'Done'
    elif x in ('pending'):
        return 'Pending'
    elif x in ('canceled'):
        return 'Canceled'
    return np.nan 

df['clean_status'] = orders['Status'].apply(clean_status)
df['clean_status']
    

0        Done
1     Pending
2        Done
3    Canceled
4        Done
Name: clean_status, dtype: str

In [ ]:
# %%
def clean_dataframe(df, column_map):
    for col, func in column_map.items():
        if col in df.columns:
            df[col] = df[col].apply(func)
    return df

mapping_1 = {
    'Full_Name' : clean_name,
    'Phone' : clean_phone,
    'City' : clean_city
}


mapping_2 = {
    'Amount' : clean_amounts,
    'Status' : clean_status
}


df_clean_1 = clean_dataframe(customers, mapping_1)
print(df_clean_1)


df_clean_2 = clean_dataframe(orders, mapping_2)
print(df_clean_2)

  Client_ID        Full_Name         Phone    City
0      C001  Charles Dickens   79991234567  Moscow
1      C002      Jane Austen           NaN     Spb
2      C003              NaN   89165553322  Moscow
3      C004     Mary Shelley  442079460958  London
4      C005      Leo Tolstoy           NaN     Spb
5      C006       Mark Twain   18001234567     NaN
  Order_ID Client_ID    Amount    Status
0     A001      C001   15000.0      Done
1     A002      C002   23000.0   Pending
2     A003      C001  500000.0      Done
3     A004      C003       NaN  Canceled
4     A005      C005    8000.0      Done
5     A006      C002   12000.0       NaN
6     A007      C007    9990.0   Pending


In [ ]:
client_full_info = df_clean_1.merge(df_clean_2, on='Client_ID')
client_full_info

,Client_ID,Full_Name,Phone,City,Order_ID,Amount,Status
0,C001,Charles Dickens,79991234567,Moscow,A001,15000.0,Done
1,C001,Charles Dickens,79991234567,Moscow,A003,500000.0,Done
2,C002,Jane Austen,NaN,Spb,A002,23000.0,Pending
3,C002,Jane Austen,NaN,Spb,A006,12000.0,NaN
4,C003,NaN,89165553322,Moscow,A004,NaN,Canceled
5,C005,Leo Tolstoy,NaN,Spb,A005,8000.0,Done


In [ ]:
df_not_dupl = client_full_info.drop_duplicates(subset=['Client_ID', 'Amount'])
df_not_dupl

,Client_ID,Full_Name,Phone,City,Order_ID,Amount,Status
0,C001,Charles Dickens,79991234567,Moscow,A001,15000.0,Done
1,C001,Charles Dickens,79991234567,Moscow,A003,500000.0,Done
2,C002,Jane Austen,NaN,Spb,A002,23000.0,Pending
3,C002,Jane Austen,NaN,Spb,A006,12000.0,NaN
4,C003,NaN,89165553322,Moscow,A004,NaN,Canceled
5,C005,Leo Tolstoy,NaN,Spb,A005,8000.0,Done
